# Lab 3 Exercise 1: Work-only evaluator

The digital twin from Lab 3, with a second LLM call that checks every reply stays strictly work-related.

- The evaluator returns a structured verdict (`is_acceptable`, `feedback`) using a Pydantic model as `response_format`.
- A rejected reply is regenerated with the evaluator's feedback added to the system prompt, up to 3 attempts, then a fallback message is returned.
- Retries run without tools, so an email is never recorded twice.

Uses the same `twin/linkedin.pdf` and `twin/summary.txt` as the main lab.

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
from pydantic import BaseModel, Field
import gradio as gr
import json

load_dotenv(override=True)
openai = OpenAI()
MODEL = "gpt-5.4-mini"

In [ ]:
reader = PdfReader("../../twin/linkedin.pdf")
linkedin = "".join(page.extract_text() or "" for page in reader.pages)

with open("../../twin/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

system_prompt = f"""You are a digital version of the person described below, chatting with visitors on their website.
Answer questions about their career, background, skills and experience, as if talking to a potential client or employer.
Steer off-topic questions back to professional topics. If you don't know the answer, say so; never make one up.
If a visitor wants to get in touch, ask for their email and record it with your tool.

## Summary
{summary}

## LinkedIn profile
{linkedin}
"""

In [ ]:
def record_email_tool(email):
    print(f"Tool called to record an email: {email}")
    with open("emails.txt", "a", encoding="utf-8") as f:
        f.write(email + "\n")
    return "Email received"

record_email_tool_json = {
    "name": "record_email_tool",
    "description": "Use this tool to record that a user provided their email address",
    "parameters": {
        "type": "object",
        "properties": {"email": {"type": "string", "description": "The email address of this user"}},
        "required": ["email"],
        "additionalProperties": False,
    },
}

tools = [{"type": "function", "function": record_email_tool_json}]

## The evaluator

In [ ]:
class Evaluation(BaseModel):
    is_acceptable: bool = Field(description="True if the reply stays strictly on work-related topics")
    feedback: str = Field(description="If not acceptable, what is off-topic and how to fix it. Otherwise a short note.")


evaluator_system_prompt = """You check replies from an AI assistant that acts as a digital twin of a professional on their website.
Decide whether the assistant's latest reply is strictly work-related.

Acceptable: career, background, skills, experience, projects, professional opportunities, getting in touch,
brief polite greetings, and politely declining an off-topic request while steering back to work.

Unacceptable: going along with off-topic requests (recipes, sports, politics, trivia, homework, unrelated code, etc.),
sharing personal details unrelated to their professional life, or dropping the role of the digital twin.

Judge the reply only: an off-topic question is fine as long as the reply does not go along with it."""


def text_of(content):
    # Gradio may send content as a string or as a list of content blocks
    if isinstance(content, str):
        return content
    return " ".join(block.get("text", "") for block in content if isinstance(block, dict))


def evaluate(reply, message, history) -> Evaluation:
    transcript = "\n".join(f"{h['role']}: {text_of(h['content'])}" for h in history) or "(no earlier messages)"
    user_prompt = f"Conversation so far:\n{transcript}\n\nLatest user message:\n{message}\n\nReply to evaluate:\n{reply}"
    messages = [
        {"role": "system", "content": evaluator_system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    response = openai.chat.completions.parse(model=MODEL, messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [ ]:
print(evaluate("I've spent most of my career building web applications.", "What kind of work do you do?", []))
print(evaluate("Sure! Layer pasta, ragu and bechamel, then bake for 45 minutes.", "Give me a lasagna recipe", []))

## The twin with the evaluator

In [ ]:
MAX_ATTEMPTS = 3
FALLBACK_REPLY = ("I'd rather keep our chat focused on my professional background. "
                  "Feel free to ask about my experience, skills or projects, or leave your email if you'd like to get in touch!")


def run_agent(messages, use_tools=True):
    extra = {"tools": tools} if use_tools else {}
    response = openai.chat.completions.create(model=MODEL, messages=messages, **extra)
    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        messages.append(message)
        for tool_call in message.tool_calls:
            email = json.loads(tool_call.function.arguments).get("email")
            record_email_tool(email)
            messages.append({"role": "tool", "content": "Email recorded", "tool_call_id": tool_call.id})
        response = openai.chat.completions.create(model=MODEL, messages=messages, **extra)
    return response.choices[0].message.content


def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    prompt = system_prompt

    for attempt in range(1, MAX_ATTEMPTS + 1):
        messages = [{"role": "system", "content": prompt}] + history + [{"role": "user", "content": message}]
        reply = run_agent(messages, use_tools=(attempt == 1))

        evaluation = evaluate(reply, message, history)
        verdict = "PASSED" if evaluation.is_acceptable else "REJECTED"
        print(f"[Evaluator] Attempt {attempt} {verdict}: {evaluation.feedback}")
        if evaluation.is_acceptable:
            return reply

        prompt = system_prompt + f"""

## Your previous answer was rejected
A quality checker rejected your last reply because it was not strictly work-related.

Rejected reply:
{reply}

Reason:
{evaluation.feedback}

Write a new reply that stays strictly on professional topics."""

    return FALLBACK_REPLY

In [ ]:
print(chat("What technologies are you strongest in?", []))
print("-" * 40)
print(chat("Can you write me a Python function to reverse a list?", []))

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)